In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import os

In [2]:
df_tickers = pd.read_csv("DIM_Tickers.csv")
lista_tickers = df_tickers["TickerYahoo"].drop_duplicates().tolist()

print(f"Tickers a procesar: {len(lista_tickers)}")
print(lista_tickers)

Tickers a procesar: 38
['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA', 'ORCL', 'PLTR', 'RDDT', 'MU', 'AMD', 'TSM', 'ALAB', 'MSTR', 'QCOM', 'YPF', 'PEP', 'UNH', 'VIST', 'SPY', 'QQQ', 'DIA', 'IWM', 'EEM', 'VEA', 'XLK', 'XLE', 'XLF', 'XLV', 'GLD', 'SLV', 'USO', 'EWZ', 'URA', 'SOXX', 'BTC-USD', 'ETH-USD']


In [ ]:
CSV_PATH = "precios_historicos.csv"

if os.path.exists(CSV_PATH):
    df_existente = pd.read_csv(CSV_PATH)
    df_existente["Date"] = pd.to_datetime(df_existente["Date"], format="mixed")
    ultima_fecha = df_existente["Date"].max()
    fecha_inicio = (ultima_fecha + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    fecha_fin    = (datetime.today() + timedelta(days=1)).strftime("%Y-%m-%d")

    # Detectar tickers nuevos (no tienen historia en el CSV existente)
    tickers_existentes = set(df_existente["Ticker"].unique())
    tickers_nuevos = [t for t in lista_tickers if t not in tickers_existentes]
    tickers_incrementales = [t for t in lista_tickers if t in tickers_existentes]

    MODO = "incremental"
    print(f"Modo: INCREMENTAL — desde {fecha_inicio} hasta {fecha_fin}")
    print(f"Registros existentes: {len(df_existente):,}")
    print(f"Tickers existentes: {len(tickers_incrementales)}")
    print(f"Tickers NUEVOS (descarga completa): {tickers_nuevos if tickers_nuevos else 'ninguno'}")
else:
    df_existente = None
    fecha_inicio = None
    fecha_fin    = None
    tickers_nuevos = lista_tickers
    tickers_incrementales = []
    MODO = "completo"
    print("Modo: COMPLETO — descargando historial de 5 años")

In [ ]:
lista_precios = []

def descargar_ticker(t, modo_ticker):
    try:
        tk = yf.Ticker(t)

        if modo_ticker == "nuevo":
            hist = tk.history(period="5y")
        else:  # incremental
            hist = tk.history(start=fecha_inicio, end=fecha_fin)

        if hist.empty:
            print(f"Sin datos nuevos para {t}")
            return None

        hist = hist.reset_index()
        hist["Ticker"] = t
        hist = hist[["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]]
        hist["Date"] = pd.to_datetime(hist["Date"]).dt.tz_localize(None)

        print(f"OK: {t} — {len(hist)} registros ({'historial completo' if modo_ticker == 'nuevo' else 'nuevos'})")
        return hist

    except Exception as e:
        print(f"Error en {t}: {e}")
        return None

# Tickers nuevos: descarga historial completo (5 años)
if tickers_nuevos:
    print(f"--- Descargando historial completo para {len(tickers_nuevos)} ticker(s) nuevo(s) ---")
    for t in tickers_nuevos:
        resultado = descargar_ticker(t, "nuevo")
        if resultado is not None:
            lista_precios.append(resultado)

# Tickers existentes: solo datos incrementales
if tickers_incrementales:
    print(f"--- Descargando datos nuevos para {len(tickers_incrementales)} ticker(s) existente(s) ---")
    for t in tickers_incrementales:
        resultado = descargar_ticker(t, "incremental")
        if resultado is not None:
            lista_precios.append(resultado)

print(f"\nTickers procesados: {len(lista_precios)} de {len(lista_tickers)}")

In [5]:
if lista_precios:
    df_nuevos = pd.concat(lista_precios, ignore_index=True)

    if MODO == "incremental" and df_existente is not None:
        df_final = pd.concat([df_existente, df_nuevos], ignore_index=True)
        df_final["Date"] = pd.to_datetime(df_final["Date"], format="mixed")
        df_final = df_final.drop_duplicates(subset=["Date", "Ticker"]).reset_index(drop=True)
    else:
        df_final = df_nuevos

    # Guardamos la fecha como string YYYY-MM-DD para evitar problemas en Power BI
    df_final["Date"] = pd.to_datetime(df_final["Date"], format="mixed").dt.strftime("%Y-%m-%d")

    df_final.to_csv(CSV_PATH, index=False)
    print(f"Archivo guardado: {CSV_PATH}")
    print(f"Total registros: {len(df_final):,}")
    print(f"Muestra de fechas: {df_final['Date'].head(3).tolist()}")
else:
    print("No hay datos nuevos — CSV sin cambios")

Archivo guardado: precios_historicos.csv
Total registros: 47,525
Muestra de fechas: ['2021-04-27', '2021-04-28', '2021-04-29']
